### Import Libraries

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print('done')

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

### Read Silver API Data as a Stream

In [2]:
silver_stream_df=(spark.readStream.format('delta')
                .option('startingVersion','latest')
                .table('Silver.api_silver_data'))
print('Silver stream configured to start from the latest Delta Version')

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 4, Finished, Available, Finished, False)

Silver stream configured to start from the latest Delta Version


### Print the Schema to check the schema of the target

In [4]:
silver_stream_df.printSchema()

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 22, Finished, Available, Finished, False)

root
 |-- Gender: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- First: string (nullable = true)
 |-- Last: string (nullable = true)
 |-- Street_Info: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Postcode: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- TimeZone_Offset: string (nullable = true)
 |-- TimeZone_Description: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- userid: string (nullable = true)
 |-- User_Name: string (nullable = true)
 |-- Password: string (nullable = true)
 |-- Salt: string (nullable = true)
 |-- MD5: string (nullable = true)
 |-- SHA1: string (nullable = true)
 |-- SHA256: string (nullable = true)
 |-- Birth_Date: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Registered_Date: string (nullable = true)
 |-- Registered_Age: string (nullable = true)
 

### Create Gold Stream from Silver Stream  DF

In [3]:
gold_stream_df=silver_stream_df.select(
    col('userid').alias('user_id'),
    col('Gender').alias('gender'),
    col('Title').alias('title'),
    col("First").alias('first_name'),
    col('Last').alias('last_name'),
    concat_ws(' ',trim(col('first')),trim(col('last'))).alias('full_name'),
    col('Email').alias('email'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Country').alias('country'),
    col('Postcode').alias('postcode'),
    col("Latitude").cast('double').alias('latitude'),
    col("Longitude").cast('double').alias('longitude'),
    col('NAT').alias('nationality'),
    col('Age').alias('age'),
    col('Birth_Date').alias('birth_date'),
    col('Registered_Date').alias('registered_date'),
    col('Registered_Age').cast('int').alias('registered_age'),
    col('Phone_Number').alias('phone_number'),
    col('Cell_Number').alias('cell_number'),
    col('Injestion_Timestamp').alias("injection_timestamp"),
    col('Processing_Timestamp').alias('processing_timestamp')

)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 5, Finished, Available, Finished, False)

### Source schema validation

In [6]:
print(gold_stream_df.printSchema())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 24, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = false)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)

None


### Function for gold customers to Upsert using Delta Tables

In [7]:
def upsert_gold_customers(batch_df,batch_id):
    latest_batch_df=batch_df.selectExpr(
        "*",
        "row_number() over(partition by user_id order by injection_timestamp desc) as rn").filter('rn==1').drop('rn')
    gold_table=DeltaTable.forName(spark,'gold.gold_customers')
    (gold_table.alias('target').merge(latest_batch_df.alias('source'),'target.user_id=source.user_id')
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(f"Batch {batch_id} processed successfully")


StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 25, Finished, Available, Finished, False)

### Gold Customers count before start of Streaming

In [8]:
display(spark.table('gold.gold_customers').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 26, Finished, Available, Finished, False)

787924

### Query for Streaming of Silver Api data to Gold Customers table

In [9]:
gold_query=(
    gold_stream_df.
    writeStream.
    foreachBatch(upsert_gold_customers).
    option('checkpointLocation','Files/checkpoints/silver_to_gold_customers').start())
print('Silver --> Gold Stream started successfully')

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 27, Finished, Available, Finished, False)

Silver --> Gold Stream started successfully


### Count of the gold_customers after the stream

In [10]:
display(spark.table('gold.gold_customers').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 28, Finished, Available, Finished, False)

787924

### Source schema creation for Data Quality table and pulling data from gold customers dataframe

In [11]:
quality_stream_df=(
    gold_stream_df
    .withColumn(
        "profile_completeness_score",
        (
            when(col('first_name').isNotNull(),1).otherwise(0)
            +
            when(col('last_name').isNotNull(),1).otherwise(0)
            +
            when(col('email').isNotNull(),1).otherwise(0)
            +
            when(col('phone_number').isNotNull(),1).otherwise(0)
            +
            when(col('country').isNotNull(),1).otherwise(0)
            +
            when(col('city').isNotNull(),1).otherwise(0)
            +
            when(col('age').isNotNull(),1).otherwise(0)
        )
    )
)

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 29, Finished, Available, Finished, False)

### Backfill for Quality

In [8]:
"""from pyspark.sql.functions import *
from delta.tables import DeltaTable
q=spark.table('gold.gold_customers')
score=(
            when(col('first_name').isNotNull(),1).otherwise(0)
            +
            when(col('last_name').isNotNull(),1).otherwise(0)
            +
            when(col('email').isNotNull(),1).otherwise(0)
            +
            when(col('phone_number').isNotNull(),1).otherwise(0)
            +
            when(col('country').isNotNull(),1).otherwise(0)
            +
            when(col('city').isNotNull(),1).otherwise(0)
            +
            when(col('age').isNotNull(),1).otherwise(0)
)
q_backfill=(
    q.withColumn("profile_completeness_score",score).
    withColumn("profile_quality",
        when(col('profile_completeness_score')==7,"Complete").
        when(col('profile_completeness_score')>=5,"Mostly Complete").
        when(col('profile_completeness_score')>=3,"Partially Complete").otherwise('Poor')
    )
)
target=DeltaTable.forName(spark,"gold.gold_data_quality")
(
    target.alias('t')
    .merge(q_backfill.alias('s'),
    "t.user_id=s.user_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("done")"""


StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 10, Finished, Available, Finished, False)

done


### Add quality category

In [12]:
quality_stream_df=(
    quality_stream_df.withColumn(
        "profile_quality",
        when(col('profile_completeness_score')==7,"Complete").
        when(col('profile_completeness_score')>=5,"Mostly Complete").
        when(col('profile_completeness_score')>=3,"Partially Complete").otherwise('Poor')
    )
)

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 30, Finished, Available, Finished, False)

### Source  quality table Schema validation

In [13]:
quality_stream_df.printSchema()

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 31, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = false)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- profile_completeness_score: integer (nullable = false)
 |-- profile_qua

### Target quality table validation

In [14]:
display(spark.table('gold.gold_data_quality').printSchema())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 32, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- profile_completeness_score: integer (nullable = true)
 |-- profile_quali

### Upsert function for Quality table

In [15]:
def upsert_quality(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    gold=DeltaTable.forName(spark,'gold.gold_data_quality')
    (gold.alias('t').merge(latest_batch_df.alias('s'),'t.user_id=s.user_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 33, Finished, Available, Finished, False)

### Quality count before the Stream

In [16]:
print(spark.table('gold.gold_data_quality').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 34, Finished, Available, Finished, False)

773753


### Start the quality stream

In [17]:
quality_query=(
    quality_stream_df.writeStream
    .foreachBatch(upsert_quality)
    .option('checkpointLocation','Files/checkpoints/quality')
    .start()
)

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 35, Finished, Available, Finished, False)

### Count of the Quality table record post Stream start

In [39]:
print(spark.table('gold.gold_data_quality').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 63, Finished, Available, Finished, False)

914510


### Quality Stream Activity Status

In [19]:
print('qulaity active',quality_query.isActive)
print('qulaity exception',quality_query.exception())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 37, Finished, Available, Finished, False)

qulaity active True
qulaity exception None


### Create Metrics Stream

In [20]:
metrics_stream_df=(
    gold_stream_df
    .withColumn('processing_latency_seconds',
    unix_timestamp('processing_timestamp')
    -
    unix_timestamp('injection_timestamp'))
)

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 38, Finished, Available, Finished, False)

### Create the Metrics Function

In [21]:
def upsert_metrics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    latest_batch_df=(
        batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    )
    table=DeltaTable.forName(
        spark,'gold.gold_pipeline_metrics'
    )
    (table.alias('target')
    .merge(
        latest_batch_df.alias('source'),
        'target.user_id=source.user_id'
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )
    print(f"Metrics batch {batch_id} Processed")

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 39, Finished, Available, Finished, False)

In [11]:
spark.table('gold.gold_pipeline_metrics').printSchema()

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 13, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- processing_latency_seconds: long (nullable = true)



In [12]:
from pyspark.sql.functions import *
m=(spark.table('gold.gold_customers')
    .withColumn('processing_latency_seconds',
    unix_timestamp('processing_timestamp')
    -
    unix_timestamp('injection_timestamp')))
m.createOrReplaceTempView("metrics_backfill")
spark.sql("""
MERGE INTO gold.gold_pipeline_metrics as t
using metrics_backfill as s
on t.user_id=s.user_id
when matched then update set *
when not matched then insert *""")
print('Metrics backfill completed')

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 14, Finished, Available, Finished, False)

Metrics backfill completed


In [13]:
print('metrics',spark.table('gold.gold_pipeline_metrics').count())

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 15, Finished, Available, Finished, False)

metrics 949449


### Metrics table record count after streaming

In [22]:
print(spark.table('gold.gold_pipeline_metrics').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 40, Finished, Available, Finished, False)

772469


### Metrics Stream Execution

In [23]:
metrics_query=(
    metrics_stream_df.writeStream
    .foreachBatch(upsert_metrics)
    .option(
        'checkpointLocation',
        'Files/checkpoints/metrics'
    )
    .start()
)

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 41, Finished, Available, Finished, False)

### Metrics table record count after streaming

In [24]:
print(spark.table('gold.gold_pipeline_metrics').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 42, Finished, Available, Finished, False)

772469


### Metrics stream activity status

In [25]:
print('qulaity active',metrics_query.isActive)
print('qulaity exception',metrics_query.exception())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 43, Finished, Available, Finished, False)

qulaity active True
qulaity exception None


### Three Streaming Gold Tables validation together

#### Counts check

In [28]:
print("customers",spark.table('gold.gold_customers').count())
print("Quality",spark.table('gold.gold_data_quality').count())
print("Metrics",spark.table('gold.gold_pipeline_metrics').count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 47, Finished, Available, Finished, False)

customers 926467
Quality 912299
Metrics 911015


#### Duplicate check

In [29]:
for t in ['gold.gold_customers','gold.gold_data_quality','gold.gold_pipeline_metrics']:
    df=spark.table(t)
    print(t,df.count()-df.select('user_id').distinct().count())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 48, Finished, Available, Finished, False)

gold.gold_customers 0
gold.gold_data_quality 0
gold.gold_pipeline_metrics 0


### All the above three streams activity status

In [30]:
for q in spark.streams.active:
    print("ID",q.id)
    print('Active',q.isActive)
    print('status',q.status)
    print('Exception',q.exception())

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 49, Finished, Available, Finished, False)

ID 97b7a74d-32f9-41e4-94a1-b20f7e8e0691
Active True
status {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Exception None
ID a21ca92f-620c-4a26-8007-dfb0c9df1690
Active True
status {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Exception None
ID afc37183-9372-4336-8867-72d0a3939297
Active True
status {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Exception None


In [31]:
for q in spark.streams.active:
    print("ID",q.id)
    print('Recent progress:',q.recentProgress[-1] if q.recentProgress else "No progress yet")
    print("-------------------")

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 50, Finished, Available, Finished, False)

ID 97b7a74d-32f9-41e4-94a1-b20f7e8e0691
Recent progress: {'id': '97b7a74d-32f9-41e4-94a1-b20f7e8e0691', 'runId': '40e2eff3-0ecd-4e6c-aeaf-ddcbca17e3bd', 'name': None, 'timestamp': '2026-09-14T09:03:42.873Z', 'batchId': 392, 'numInputRows': 36, 'inputRowsPerSecond': 240.0, 'processedRowsPerSecond': 3.1743232519178206, 'durationMs': {'addBatch': 9991, 'commitOffsets': 194, 'getBatch': 190, 'latestOffset': 731, 'queryPlanning': 13, 'triggerExecution': 11341, 'walCommit': 220}, 'stateOperators': [], 'sources': [{'description': 'DeltaSource[abfss://87af2989-49a5-40e9-960f-2da392c0bac9@onelake.dfs.fabric.microsoft.com/d037bf72-b0fa-4785-92a9-84df50cd03e7/Tables/Silver/api_silver_data]', 'startOffset': {'sourceVersion': 1, 'reservoirId': '93568c3d-dbf8-4791-9910-39b900672592', 'reservoirVersion': 993, 'index': -1, 'isStartingVersion': False}, 'endOffset': {'sourceVersion': 1, 'reservoirId': '93568c3d-dbf8-4791-9910-39b900672592', 'reservoirVersion': 994, 'index': -1, 'isStartingVersion': Fals

### Pre validate the demographic and demographics table counts

In [34]:
print('Demographics rows',spark.table('gold.gold_demographics').count())
print('Demographics state rows',spark.table('gold.gold_demographic_state').count())
print('Active Streams',len(spark.streams.active))

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 53, Finished, Available, Finished, False)

Demographics rows 168
Demographics state rows 504608
Active Streams 3


### Create the Demographics Stream

In [79]:
demographics_stream_df=(
            spark.readStream.format('delta').
            table('silver.api_silver_data')
            .select(
                col('userid').alias('user_id'),
                col('Gender').alias('gender'),
                col('Age').alias('age'),
                col('NAT').alias('nationality'),
                col("Injestion_Timestamp").alias('Injection_Timestamp')
            ))

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 81, Finished, Available, Finished, False)

### Helper table creation once 

In [52]:
"""state_df=(
    spark.table('gold.gold_customers')
    .select('user_id','gender','age','nationality')
    .withColumn('age_group',
    when(col('age')<18,'Under 18').
    when(col('age')<=25,'18-25').
    when(col('age')<=35,'26-35').
    when(col('age')<=50,'36-50').
    otherwise('51+')
    ).drop('age')
)"""

StatementMeta(, 7f7c3499-d886-4bda-b530-95f3b7fff8b6, 76, Finished, Available, Finished, False)

"state_df=(\n    spark.table('gold.gold_customers')\n    .select('user_id','gender','age','nationality')\n    .withColumn('age_group',\n    when(col('age')<18,'Under 18').\n    when(col('age')<=25,'18-25').\n    when(col('age')<=35,'26-35').\n    when(col('age')<=50,'36-50').\n    otherwise('51+')\n    ).drop('age')\n)"

### Overwrite once for the helper table

In [31]:
"""state_df.write.mode('overwrite').format("delta").saveAsTable('gold.gold_demographic_state')"""

StatementMeta(, 3b018153-6f6d-4197-9851-ae1098bcaa8c, 50, Finished, Available, Finished, False)

### Incremental Function for demographic state Helper Table

In [80]:
def update_demographics(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(
    batch_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn')
    .withColumn(
    'age_group',
    when(col('age')<18,'Under 18').
    when(col('age')<=25,'18-25').
    when(col('age')<=35,'26-35').
    when(col('age')<=50,'36-50').
    otherwise('51+'))
    .select('user_id','gender','age_group','nationality'))
    state=DeltaTable.forName(spark,'gold.gold_demographic_state')
    changes=(
            new_df.alias('n').join(state.toDF().alias('s'),'user_id','left')
            .select(
                col('s.gender').alias('old_gender'),
                col('s.age_group').alias('old_age_group'),
                col('s.nationality').alias('old_nationality'),
                col('n.gender').alias('new_gender'),
                col('n.age_group').alias('new_age_group'),
                col('n.nationality').alias('new_nationality')
                )
            )

    old_changes=(changes
            .filter(col('old_gender').isNotNull())
            .select(
                col('old_gender').alias('gender'),
                col('old_age_group').alias('age_group'),
                col('old_nationality').alias('nationality'),
                lit(-1).alias('delta')
            ))

    new_changes=(changes
            .filter(col('new_gender').isNotNull())
            .select(
                col('new_gender').alias('gender'),
                col('new_age_group').alias('age_group'),
                col('new_nationality').alias('nationality'),
                lit(1).alias('delta')
            ))

    deltas=(
        old_changes.
        unionByName(new_changes)
        .groupBy('gender','age_group','nationality')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )

    demo=DeltaTable.forName(spark,'gold.gold_demographics')
    (
        demo.alias('t')
        .merge(deltas.alias('s'),
        """t.gender=s.gender AND t.age_group=s.age_group AND t.nationality=s.nationality""")
        .whenMatchedUpdate(set={'customer_count':"t.customer_count+s.delta"})
        .whenNotMatchedInsert(values={'gender':'s.gender','age_group':'s.age_group',
        'nationality':'s.nationality',
        'customer_count':'s.delta'})
        .execute())
    (
        state.alias('t')
        .merge(new_df.alias('s'),
            't.user_id = s.user_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Demographic batch {batch_id} processed")


StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 82, Finished, Available, Finished, False)

### Start the Demographics stream

In [81]:
demographics_query=(
    demographics_stream_df.writeStream
    .foreachBatch(update_demographics)
    .option('checkpointLocation','Files/checkpoints/demographics')
    .start()
)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 83, Finished, Available, Finished, False)

### Demographics query activity status

In [84]:
print('demographics active',demographics_query.isActive)
print('Demographics exception',demographics_query.exception())

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 86, Finished, Available, Finished, False)

demographics active True
Demographics exception None


### Stream Post Validations for Demographics

### Count Check for the state and demographics tables

In [87]:
print('geotable',spark.sql('select sum(customer_count) from gold.gold_demographics').first()[0])
print('state',spark.table('gold.gold_demographic_state').count())

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 89, Finished, Available, Finished, False)

geotable 933340
state 933340


### Create Geography state for aggreagating and merging to micro batches as a Helper Table run once Queries

In [7]:
"""geo_state_df=(
    spark.table('gold.gold_customers')
    .select('user_id','country','state','city')
)
geo_state_df.write.format('delta').mode('overwrite').saveAsTable('gold.gold_geography_state')"""

StatementMeta(, 3b018153-6f6d-4197-9851-ae1098bcaa8c, 26, Finished, Available, Finished, False)

#### Count of the records in the helper table for geography

In [58]:
print(spark.table('gold.gold_geography_state').count())

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 60, Finished, Available, Finished, False)

933340


### Geography stream defination

In [50]:
geography_stream_df=(
    spark.readStream
    .format('delta')
    .table('silver.api_silver_data')
    .select(
        col('userid').alias('user_id'),
        col('Country').alias('country'),
        col('State').alias('state'),
        col('City').alias('city'),
        col("Injestion_Timestamp").alias('injection_timestamp')
    )
)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 52, Finished, Available, Finished, False)

### Upsert function for geography and state table

In [52]:
def update_geography(batch_df,batch_id):
    window_spec=Window.partitionBy('user_id').orderBy(col('injection_timestamp').desc())
    new_df=(batch_df
            .withColumn('rn',row_number().over(window_spec))
            .filter(col('rn')==1)
            .drop('rn')
            .select('user_id','country','state','city'))
    state=DeltaTable.forName(spark,'gold.gold_geography_state')
    changes=(
        new_df.alias('n')
        .join(state.toDF().alias('s'),'user_id','left')
        .select(
            col('s.country').alias("old_country"),
            col('s.state').alias('old_state'),
            col('s.city').alias('old_city'),
            col('n.country').alias("new_country"),
            col('n.state').alias('new_state'),
            col('n.city').alias('new_city')
        )
    )
    old_changes=(
        changes.filter(col('old_country').isNotNull())
        .select(
            col('old_country').alias('country'),
            col('old_state').alias('state'),
            col('old_city').alias('city'),
            lit(-1).alias('delta')
        )
    )
    new_changes=(
        changes.filter(col('new_country').isNotNull())
        .select(
            col('new_country').alias('country'),
            col('new_state').alias('state'),
            col('new_city').alias('city'),
            lit(1).alias('delta')
        )
    )
    deltas=(
        old_changes
        .unionByName(new_changes)
        .groupBy('country','state','city')
        .sum('delta')
        .withColumnRenamed('sum(delta)','delta')
    )
    geo=DeltaTable.forName(
        spark,'gold.gold_geography'
    )
    (geo.alias('t')
    .merge(deltas.alias('s'),
        "t.country=s.country AND t.state=s.state AND t.city=s.city")
        .whenMatchedUpdate(set={"customer_count":'t.customer_count+s.delta'})
        .whenNotMatchedInsert(
            values={"country":"s.country",
                    "state":"s.state",
                    "city":"s.city",
                    "customer_count":"s.delta"}
        ).execute()
    )
    (state.alias('t')
    .merge(new_df.alias('s'),"t.user_id=s.user_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
    print(f"Geography batch {batch_id} processed")

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 54, Finished, Available, Finished, False)

### Start Geography Stream

In [53]:
geography_query=(
    geography_stream_df.writeStream
    .foreachBatch(update_geography)
    .option('checkpointLocation','Files/checkpoints/geography')
    .start()
)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 55, Finished, Available, Finished, False)

### Geography Stream Activity Status

In [55]:
print(geography_query.isActive)
print(geography_query.exception())

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 57, Finished, Available, Finished, False)

True
None


### Geography Post Steam validation

In [59]:
geo_check=spark.sql("""
    select sum(customer_count) as Total_Customer_records,
    count(*) as geography_groups,
    sum(case when customer_count<0 then 1 else 0 end) as neg_groups from gold.gold_geography""")
display(geo_check)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 050fc16e-c500-4654-a2ec-04b60cf8a807)

In [61]:
print('customer',gold_query.id)
print('Qulaity',quality_query.id)
print('Metric',metrics_query.id)
print('demographics',demographics_query.id)
print('geography',geography_query.id)

StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 63, Finished, Available, Finished, False)

NameError: name 'gold_query' is not defined

In [ ]:
print('Bronze table total records',spark.table('bronze.api_raw_data').count())
print('Silver totals',spark.table('silver.api_silver_data').count())
print("Customers",spark.table('gold.gold_customers').count())
print("Quality",spark.table('gold.gold_data_quality').count())
print("Metrics",spark.table('gold.gold_pipeline_metrics').count())
print('Demography table',spark.sql('select sum(customer_count) from gold.gold_demographics').first()[0])
print('Demography state',spark.table('gold.gold_demographic_state').count())
print('Geography table',spark.sql('select sum(customer_count) from gold.gold_geography').first()[0])
print('Geography state',spark.table('gold.gold_geography_state').count())
print('Duplicate customer IDs:',spark.sql('select count(*)-count(distinct user_id) from gold.gold_customers').first()[0])


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [CapacityLimitExceeded] Unable to complete the action because your organization’s Fabric compute capacity has exceeded its limits. Try again later. HTTP status code: 429.

In [1]:
print('Bronze table total records',spark.table('bronze.api_raw_data').count())
print('Silver totals',spark.table('silver.api_silver_data').count())
print("Customers",spark.table('gold.gold_customers').count())
print("Quality",spark.table('gold.gold_data_quality').count())
print("Metrics",spark.table('gold.gold_pipeline_metrics').count())
print('Demography table',spark.sql('select sum(customer_count) from gold.gold_demographics').first()[0])
print('Demography state',spark.table('gold.gold_demographic_state').count())
print('Geography table',spark.sql('select sum(customer_count) from gold.gold_geography').first()[0])
print('Geography state',spark.table('gold.gold_geography_state').count())
print('Duplicate customer IDs:',spark.sql('select count(*)-count(distinct user_id) from gold.gold_customers').first()[0])


StatementMeta(, a18b971e-c4f5-420b-ad36-3fe303ce3482, 3, Finished, Available, Finished, False)

Bronze table total records 960421
Silver totals 960158
Customers 953468
Quality 953468
Metrics 953468
Demography table 953353
Demography state 953353
Geography table 953353
Geography state 953353
Duplicate customer IDs: 0


### Demographics reset 

In [76]:
"""
spark.sql('drop table if exists gold.gold_demographics')
spark.sql('drop table if exists gold.gold_demographic_state')
spark.sql(create table gold.gold_demographics(
    gender string,
    age_group string,
    nationality string,
    customer_count bigint
)using DELTA)
spark.sql(create table gold.gold_demographic_state(
    user_id string,
    gender string,
    age_group string,
    nationality string
)using DELTA)
print('demographics tables reset successfully')
"""



StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 78, Finished, Available, Finished, False)

demographics tables reset successfully


### Geography reset

In [64]:
"""
spark.sql('drop table if exists gold.gold_geography')
spark.sql('drop table if exists gold.gold_geography_state')
spark.sql('create table gold.gold_geography(
    country string,
    state string,
    city string,
    customer_count bigint
)using DELTA')
spark.sql('create table gold.gold_geography_state(
    user_id string,
    country string,
    state string,
    city string
)using DELTA')
print('demographics tables reset successfully')

"""




StatementMeta(, 71d7fc18-72c8-4ed1-a2b3-08b3a40092a4, 66, Finished, Available, Finished, False)

"\nspark.sql('drop table if exists gold.gold_geography')\nspark.sql('drop table if exists gold.gold_geography_state')\nspark.sql('create table gold.gold_geography(\n    country string,\n    state string,\n    city string,\n    customer_count bigint\n)using DELTA')\nspark.sql('create table gold.gold_geography_state(\n    user_id string,\n    country string,\n    state string,\n    city string\n)using DELTA')\nprint('demographics tables reset successfully')\n\n"

In [5]:
q=spark.table('gold.gold_customers')
q.printSchema()

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 7, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)



In [6]:
spark.table('gold.gold_data_quality').printSchema()

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 8, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- nationality: string (nullable = true)
 |-- age: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- registered_date: string (nullable = true)
 |-- registered_age: integer (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- cell_number: string (nullable = true)
 |-- injection_timestamp: timestamp (nullable = true)
 |-- processing_timestamp: timestamp (nullable = true)
 |-- profile_completeness_score: integer (nullable = true)
 |-- profile_quali

In [14]:
spark.sql("OPTIMIZE bronze.api_raw_data")

StatementMeta(, d8a18024-507f-4682-947d-1fe731859aa3, 16, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,